# 05 - アーリーゲーム分析

15分時点のゴールド差・CS差がその後の勝率にどう影響するかを分析する。
ファーストブラッド・ファーストタワーの影響度も算出する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import yaml
from pathlib import Path
from sklearn.linear_model import LogisticRegression

for font in ['Meiryo', 'Yu Gothic', 'MS Gothic', 'Hiragino Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except ValueError:
        continue
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

DATA = Path('..') / 'data' / 'processed'
CONFIG = Path('..') / 'config' / 'settings.yaml'

with open(CONFIG, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
MEMBERS = {f'{m["game_name"]}#{m["tag_line"]}' for m in cfg['members']}

df_frames = pd.read_csv(DATA / 'timeline_frames.csv')
df_events = pd.read_csv(DATA / 'timeline_events.csv')
df_matches = pd.read_csv(DATA / 'matches.csv', parse_dates=['gameCreation'])
df_players = pd.read_csv(DATA / 'player_stats.csv')

df_frames['riotId'] = df_frames['summonerName'] + '#' + df_frames['tagLine']
df_frames['isMember'] = df_frames['riotId'].isin(MEMBERS)

In [ ]:
# 15分時点のチームゴールド差を算出
TARGET_MIN = 15.0

frames_15 = df_frames[df_frames['timestampMin'] == TARGET_MIN].copy()

team_gold_15 = frames_15.groupby(['matchId', 'teamId'])['totalGold'].sum().unstack(fill_value=0)
team_gold_15.columns = ['team100_gold', 'team200_gold']
team_gold_15['goldDiff_100'] = team_gold_15['team100_gold'] - team_gold_15['team200_gold']

team_gold_15 = team_gold_15.reset_index().merge(
    df_matches[['matchId', 'team100Win']], on='matchId', how='left'
)

print(f'15分データのある試合数: {len(team_gold_15)}')
team_gold_15.head()

In [ ]:
# 15分ゴールド差 vs 勝率: 散布図 + ロジスティック回帰
X = team_gold_15[['goldDiff_100']].values
y = team_gold_15['team100Win'].astype(int).values

if len(X) >= 10:
    model = LogisticRegression()
    model.fit(X, y)

    x_range = np.linspace(X.min() - 1000, X.max() + 1000, 300)
    y_pred = model.predict_proba(x_range.reshape(-1, 1))[:, 1]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(X, y, alpha=0.3, s=20, label='試合結果')
    ax.plot(x_range, y_pred, 'r-', linewidth=2, label='ロジスティック回帰')
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('15分時点 チームゴールド差 (Team100 - Team200)')
    ax.set_ylabel('Team100 勝率')
    ax.set_title('15分時点ゴールド差 vs 勝率（ロジスティック回帰）')
    ax.legend()

    for gd in [-3000, -2000, -1000, 0, 1000, 2000, 3000]:
        prob = model.predict_proba([[gd]])[0][1]
        ax.annotate(f'{gd:+d}g → {prob*100:.0f}%', xy=(gd, prob),
                    fontsize=8, ha='center', va='bottom')

    plt.tight_layout()
    plt.show()
else:
    print('データが不足しています（10試合未満）')

In [ ]:
# ゴールド差の閾値分析: ビン別勝率
bins = [-np.inf, -5000, -3000, -2000, -1000, 0, 1000, 2000, 3000, 5000, np.inf]
labels = ['<-5k', '-5k~-3k', '-3k~-2k', '-2k~-1k', '-1k~0', '0~1k', '1k~2k', '2k~3k', '3k~5k', '>5k']

team_gold_15['goldBin'] = pd.cut(team_gold_15['goldDiff_100'], bins=bins, labels=labels)

bin_wr = team_gold_15.groupby('goldBin', observed=False).agg(
    games=('team100Win', 'count'),
    winrate=('team100Win', 'mean'),
).reset_index()
bin_wr['winrate_pct'] = (bin_wr['winrate'] * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if wr < 50 else '#2ecc71' for wr in bin_wr['winrate_pct']]
bars = ax.bar(bin_wr['goldBin'].astype(str), bin_wr['winrate_pct'], color=colors)
ax.axhline(y=50, color='gray', linestyle='--')
ax.set_xlabel('15分時点 ゴールド差ビン')
ax.set_ylabel('勝率 %')
ax.set_title('15分時点ゴールド差ビン別 勝率')

for bar, games in zip(bars, bin_wr['games']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'n={games}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# メンバー別 レーン15分ゴールド差（対面との差）
member_15 = frames_15[frames_15['isMember'] & frames_15['goldDiffVsOpponent'].notna()].copy()

if not member_15.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=member_15, x='summonerName', y='goldDiffVsOpponent',
                hue='win', ax=ax, palette={True: '#2ecc71', False: '#e74c3c'})
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title('メンバー別 15分時点ゴールド差（vs 対面）')
    ax.set_xlabel('')
    ax.set_ylabel('ゴールド差')
    ax.legend(title='勝利')
    plt.tight_layout()
    plt.show()

    median_diff = member_15.groupby('summonerName')['goldDiffVsOpponent'].median().round(0)
    print('\nメンバー別 15分ゴールド差中央値:')
    print(median_diff.sort_values(ascending=False))
else:
    print('メンバーの15分データがありません')

In [ ]:
# 15分CS差 vs 勝率
member_15_cs = frames_15[frames_15['isMember'] & frames_15['csDiffVsOpponent'].notna()].copy()

if not member_15_cs.empty:
    cs_bins = [-np.inf, -30, -15, 0, 15, 30, np.inf]
    cs_labels = ['<-30', '-30~-15', '-15~0', '0~15', '15~30', '>30']
    member_15_cs['csBin'] = pd.cut(member_15_cs['csDiffVsOpponent'], bins=cs_bins, labels=cs_labels)

    cs_wr = member_15_cs.groupby('csBin', observed=False).agg(
        games=('win', 'count'),
        winrate=('win', 'mean'),
    ).reset_index()
    cs_wr['winrate_pct'] = (cs_wr['winrate'] * 100).round(1)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#e74c3c' if wr < 50 else '#2ecc71' for wr in cs_wr['winrate_pct']]
    ax.bar(cs_wr['csBin'].astype(str), cs_wr['winrate_pct'], color=colors)
    ax.axhline(y=50, color='gray', linestyle='--')
    ax.set_xlabel('15分時点 CS差（vs 対面）')
    ax.set_ylabel('勝率 %')
    ax.set_title('15分CS差ビン別 勝率（メンバー全体）')
    plt.tight_layout()
    plt.show()

In [ ]:
# ファーストブラッド / ファーストタワー の勝率インパクト
df_p = pd.read_csv(DATA / 'player_stats.csv')
df_p['riotId'] = df_p['summonerName'] + '#' + df_p['tagLine']
df_p_m = df_p[df_p['riotId'].isin(MEMBERS)].copy()

fb_games = df_p_m.groupby('matchId').agg(
    fb_taken=('firstBloodKill', 'any'),
    ft_taken=('firstTowerKill', 'any'),
    win=('win', 'first'),
).reset_index()

impact_data = []
for event, col in [('ファーストブラッド', 'fb_taken'), ('ファーストタワー', 'ft_taken')]:
    for taken in [True, False]:
        subset = fb_games[fb_games[col] == taken]
        impact_data.append({
            'イベント': event,
            '取得': '取得' if taken else '非取得',
            '試合数': len(subset),
            '勝率': round(subset['win'].mean() * 100, 1) if len(subset) > 0 else 0,
        })

df_impact = pd.DataFrame(impact_data)

fig, ax = plt.subplots(figsize=(8, 4))
pivot = df_impact.pivot(index='イベント', columns='取得', values='勝率')
pivot.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('勝率 %')
ax.set_title('ファーストブラッド / ファーストタワー 取得時の勝率差')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
plt.legend(title='')
plt.tight_layout()
plt.show()

print(df_impact.to_string(index=False))